# 분주한 승강씨 — AI 스프라이트 자동 생성

**한 셀로 끝.** 아래 셀 하나만 실행하면 의존성 설치 → 모델 로딩 → 51개 스프라이트 생성 → zip 다운로드 까지 자동.

**런타임 설정**: 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** (또는 더 빠른 것).

전체 소요 시간 (T4 기준): **약 25~35분** (51개 × 평균 30초 + 셋업 5분).

끝나면 `sprites_out.zip` 이 자동 다운로드됨. 압축 풀어서 `public/sprites/` 에 복사 → 게임 새로고침 → 자동 적용.

**선택**: 마음에 안 드는 키만 재생성하고 싶으면 맨 아래 Gradio UI 셀 실행.

In [ ]:
# ============================================================
# 원셀: 의존성 → 모델 → 일괄 생성 → zip → 다운로드
# ============================================================

# ── 1) 의존성 ────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers', 'transformers', 'accelerate', 'safetensors', 'peft', 'rembg', 'pillow', 'gradio'],
    check=True)

import os, shutil, torch
from PIL import Image
from tqdm.notebook import tqdm
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from rembg import remove

# ── 2) 프롬프트 카탈로그 (sprite-prompts.ts 에서 추출) ────
COMMON_SUFFIX = ', pixel art, sharp pixel grid, transparent background, 16-color palette, ENDESGA-16 style, retro indie game asset, front view'

SPRITE_PROMPTS = {
    'elevator-cab':         'small retro elevator cab interior, metallic blue panels, yellow accent strip at bottom, small floor display screen with up arrow at top, 64x96 px, vertical rectangular box, two-tone shading',
    'elevator-cab-broken':  'broken elevator cab, sparking wires, red warning light, cracked panels, smoke wisp, exclamation mark inside, 64x96 px, vertical box, distressed look',
    'elevator-door-open':   'elevator doors fully opened, split horizontally, brushed metal panels, thin shadow inside, top-down compressed view, 64x16 px wide horizontal strip',
    'elevator-door-closed': 'elevator doors closed, dark center seam, metallic panels with vertical grooves, two-tone gray, 64x96 px, tall rectangular',
    'elevator-cable':       'thin vertical elevator cable, dark gray steel rope, simple tileable strip, 4 px wide repeating vertical line',
    'passenger-normal':       'simple pixel character, casual office worker, plain shirt, neutral pose standing, ordinary appearance, 16x24 px, head + body + small legs',
    'passenger-vip':          'pixel character VIP businessman, golden suit accent, confident posture, slight smirk, premium look, 16x24 px, head + body',
    'passenger-elderly':      'pixel elderly character with cane, gray hair, hunched posture, purple cardigan, slow gentle look, 16x24 px',
    'passenger-suit':         'pixel businessman in blue suit and tie, briefcase, sharp posture, hurried look, 16x24 px',
    'passenger-group':        'pixel three friends standing together as group, orange shirts, casual chatty pose, side by side trio, 16x24 px',
    'passenger-baggage':      'pixel character carrying large suitcase, brown coat, big bag taking space, 20x28 px, wide silhouette',
    'passenger-shady':        'pixel suspicious character in dark hoodie, hood up, dark red tones, shifty pose, 16x24 px',
    'passenger-tourist':      'pixel tourist with camera, green shirt, sun hat, bright cheerful look, 16x24 px',
    'passenger-staff':        'pixel maintenance staff in gray work uniform, name tag, neutral pose, 16x24 px',
    'passenger-thief':        'pixel thief character, black hoodie with red mask accent, sneaky crouching posture, eyes shifted sideways, 16x24 px',
    'passenger-patient':      'pixel hospital patient in light pink gown, IV pole or bandage, weak posture, 16x24 px, soft pale palette',
    'passenger-medical':      'pixel doctor in white coat with red cross accent, stethoscope, calm professional pose, 16x24 px',
    'passenger-hotel-guest':  'pixel hotel guest with rolling suitcase, beige coat, comfortable travel look, 20x28 px',
    'passenger-crew':         'pixel airline crew member, navy blue uniform, scarf, professional flight attendant pose, 16x24 px',
    'floor-lobby':       'flat icon symbol for hotel lobby, sofa or chandelier silhouette, gray-blue palette, simple emblem, 32x32 px, dark rounded square background',
    'floor-office':      'flat icon symbol for office floor, desk with monitor silhouette, blue palette, professional emblem, 32x32 px, dark rounded square background',
    'floor-restaurant':  'flat icon symbol for restaurant, fork and spoon silhouette, orange palette, simple emblem, 32x32 px, dark rounded square background',
    'floor-rooftop':     'flat icon symbol for rooftop, sun and small antenna silhouette, cyan palette, simple emblem, 32x32 px, dark rounded square',
    'floor-basement':    'flat icon symbol for basement, downward arrow with brick texture, dark gray palette, simple emblem, 32x32 px',
    'floor-gym':         'flat icon symbol for gym, dumbbell silhouette, green palette, simple emblem, 32x32 px, dark rounded square',
    'floor-mall':        'flat icon symbol for shopping mall, shopping bag silhouette, pink palette, simple emblem, 32x32 px',
    'floor-medical':     'flat icon symbol for medical floor, red cross on white background, hospital emblem, 32x32 px, clean and clinical',
    'floor-hotel-room':  'flat icon symbol for hotel room, bed silhouette, beige palette, simple emblem, 32x32 px, dark rounded square',
    'floor-gate':        'flat icon symbol for airport gate, airplane silhouette, blue palette, departure board feeling, 32x32 px',
    'floor-checkin':     'flat icon symbol for check-in counter, podium with luggage, brown palette, simple emblem, 32x32 px',
    'env-subway':        'pixel subway entrance, dark stairwell going down with metal railings, M sign above, 48x32 px, side-view perspective',
    'env-escalator':     'pixel escalator going up, diagonal metal steps, side view with handrail, green safety strip, 32x64 px, animated feel',
    'env-stairs':        'pixel stairs going up, side view, simple gray steps, 32x64 px, neutral lighting',
    'env-helipad':       'pixel helipad on rooftop, white H symbol on dark square, faded landing markings, 64x32 px, top-down view',
    'env-toilet-clean':  'pixel bathroom symbol clean and shiny, white tile background, blue droplet sparkle, 24x24 px',
    'env-toilet-dirty':  'pixel bathroom symbol dirty and gross, sickly green-brown tones, fly or stink lines, 24x24 px, distressed',
    'ui-icon-gold':       'pixel gold coin icon, bright yellow with G letter or shiny edge, simple circular emblem, 16x16 px, dark background',
    'ui-icon-anger':      'pixel anger icon, red exclamation mark with sharp edges, alarmed look, 16x16 px, dark background',
    'ui-icon-clock':      'pixel clock icon, circular face with hour/minute hands, neutral gray, 16x16 px, dark background',
    'ui-icon-passenger':  'pixel person silhouette icon, generic standing figure, neutral gray, 16x16 px, dark background',
    'ui-icon-elevator':   'pixel elevator icon, simple box with up/down arrow, blue accent, 16x16 px, dark background',
    'decor-wall-tile':       'pixel building exterior wall tile, dark gray concrete texture, subtle dotted pattern, 32x32 px, seamlessly tileable',
    'decor-window-lit':      'pixel building window lit at night, warm yellow square divided by cross frame, 8x12 px, glowing feel',
    'decor-window-dark':     'pixel building window unlit, dark gray rectangle divided by cross frame, 8x12 px, empty office at night',
    'decor-title-building':  'pixel art tall city building silhouette at night, many small glowing windows, retro 8-bit style, 256x480 px, vertical composition, Seoul skyline mood',
    'character-mentor-default':  'visual novel anime portrait, korean man in late 30s, gray business suit, slightly cynical neutral expression, half-body shot, 256x384 px, soft anime line art, transparent background, slight stubble, knowing tired eyes, building manager vibe',
    'character-mentor-smirk':    'visual novel anime portrait, same korean man in late 30s, gray business suit, smirking sardonic expression, half-body shot, 256x384 px, anime style, transparent background, raised eyebrow, condescending grin',
    'character-owner-default':   'visual novel anime portrait, korean man in 50s, black premium business suit, stern serious expression, half-body shot, 256x384 px, anime style, transparent background, gray streaks in hair, authoritative aura, building owner vibe',
    'character-owner-angry':     'visual novel anime portrait, same korean man in 50s, black suit, furious expression with frowning brow, half-body shot, 256x384 px, anime style, transparent background, red anger marks, intense glare',
    'character-player-default':  'visual novel anime portrait, korean young adult in 20s, white shirt with loosened tie, slightly nervous neutral expression, half-body shot, 256x384 px, anime style, transparent background, freshly hired employee vibe, slight bags under eyes',
    'character-player-worried':  'visual novel anime portrait, same korean young adult in 20s, white shirt, worried anxious expression, half-body shot, 256x384 px, anime style, transparent background, hand near face, concerned eyebrows, sweat drop',
}

SPRITE_SIZES = {
    'elevator-cab': (64, 96), 'elevator-cab-broken': (64, 96),
    'elevator-door-open': (64, 16), 'elevator-door-closed': (64, 96),
    'elevator-cable': (4, 96),
    **{f'passenger-{k}': (16, 24) for k in ['normal','vip','elderly','suit','group','shady','tourist','staff','thief','patient','medical','crew']},
    'passenger-baggage': (20, 28), 'passenger-hotel-guest': (20, 28),
    **{f'floor-{k}': (32, 32) for k in ['lobby','office','restaurant','rooftop','basement','gym','mall','medical','hotel-room','gate','checkin']},
    'env-subway': (48, 32), 'env-escalator': (32, 64), 'env-stairs': (32, 64),
    'env-helipad': (64, 32), 'env-toilet-clean': (24, 24), 'env-toilet-dirty': (24, 24),
    **{f'ui-icon-{k}': (16, 16) for k in ['gold','anger','clock','passenger','elevator']},
    'decor-wall-tile': (32, 32), 'decor-window-lit': (8, 12), 'decor-window-dark': (8, 12),
    'decor-title-building': (256, 480),
    **{k: (256, 384) for k in SPRITE_PROMPTS if k.startswith('character-')},
}

def is_char(k): return k.startswith('character-')
def full_prompt(k): return SPRITE_PROMPTS[k] if is_char(k) else SPRITE_PROMPTS[k] + COMMON_SUFFIX

print(f'프롬프트 {len(SPRITE_PROMPTS)}개 로드')

# ── 3) SDXL + 픽셀 LoRA 로딩 ────────────────────
print('SDXL 모델 로딩 중... (5~10분, 최초 한 번)')
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    vae=vae, torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
).to('cuda')
pipe.load_lora_weights('nerijs/pixel-art-xl', adapter_name='pixel')
pipe.set_adapters([])
print('모델 로딩 완료')

# ── 4) 생성 함수 ─────────────────────────────────
OUT_DIR = '/content/sprites_out'
os.makedirs(OUT_DIR, exist_ok=True)

def generate(key, prompt_override=None, seed=None):
    prompt = prompt_override or full_prompt(key)
    char = is_char(key)
    pipe.set_adapters([] if char else ['pixel'], adapter_weights=None if char else [1.0])
    gen = torch.Generator('cuda').manual_seed(seed) if seed is not None else None
    img = pipe(
        prompt=prompt,
        negative_prompt='blurry, low quality, watermark, signature, text, ugly, deformed' if char else 'blurry, anti-aliasing, smooth, gradient, photorealistic',
        num_inference_steps=24, guidance_scale=6.5 if char else 8.0,
        height=1024, width=1024, generator=gen,
    ).images[0]
    img = remove(img)
    tw, th = SPRITE_SIZES.get(key, (64, 64))
    return img.resize((tw, th), Image.LANCZOS if char else Image.NEAREST)

# ── 5) 일괄 생성 ─────────────────────────────────
SKIP_EXISTING = True
failed = []
for key in tqdm(list(SPRITE_PROMPTS.keys()), desc='생성 중'):
    p = os.path.join(OUT_DIR, f'{key}.png')
    if SKIP_EXISTING and os.path.exists(p): continue
    try:
        generate(key).save(p)
    except Exception as e:
        print(f'  ✗ {key}: {e}')
        failed.append(key)

print(f'\n완료 — {len(os.listdir(OUT_DIR))}/{len(SPRITE_PROMPTS)} 생성됨')
if failed: print(f'실패: {failed}')

# ── 6) zip + 자동 다운로드 ──────────────────────
shutil.make_archive('/content/sprites_out', 'zip', OUT_DIR)
print('\nsprites_out.zip 준비 완료')
try:
    from google.colab import files
    files.download('/content/sprites_out.zip')
    print('다운로드 시작됨. 압축 풀어서 public/sprites/ 에 복사 후 게임 새로고침.')
except Exception:
    print('수동 다운로드: 좌측 파일 패널 → sprites_out.zip 우클릭 → 다운로드')

---

## 선택사항 — Gradio UI 로 재생성

마음에 안 드는 키만 골라서 프롬프트 수정 + 시드 변경 + 재생성. 결과 즉시 미리보기.

**주의**: 위 셀이 완료된 후에만 실행 가능 (모델/프롬프트 로드되어 있어야 함).

In [ ]:
import gradio as gr

def ui_load(key): return full_prompt(key)

def ui_generate(key, prompt, seed):
    seed_val = int(seed) if seed and str(seed).strip() else None
    img = generate(key, prompt_override=prompt, seed=seed_val)
    img.save(os.path.join(OUT_DIR, f'{key}.png'))
    preview = img.resize((img.width * 4, img.height * 4), Image.NEAREST if not is_char(key) else Image.LANCZOS)
    return preview, f'✓ 저장 → {key}.png ({img.width}×{img.height})'

with gr.Blocks(title='Sprite Regen') as demo:
    gr.Markdown('### 스프라이트 재생성')
    with gr.Row():
        key_dd = gr.Dropdown(list(SPRITE_PROMPTS.keys()), value=list(SPRITE_PROMPTS.keys())[0], label='Sprite 키')
        seed_in = gr.Textbox(label='Seed (빈값=랜덤)', value='', scale=0)
    prompt_box = gr.Textbox(label='프롬프트', lines=4)
    btn = gr.Button('생성', variant='primary')
    out_img = gr.Image(label='미리보기 (4배 확대)')
    out_status = gr.Markdown()
    key_dd.change(ui_load, key_dd, prompt_box)
    btn.click(ui_generate, [key_dd, prompt_box, seed_in], [out_img, out_status])
    demo.load(ui_load, key_dd, prompt_box)

demo.launch(share=True)